# Pelican 



In [1]:
import os
import numpy as np
import numpy.ma as ma
import matplotlib.pyplot as plt
from scipy.stats import norm
from astropy.io import fits
import astropy.units as u

from dysh.fits.gbtfitsload import GBTFITSLoad
from dysh.fits.gbtfitsload import GBTOffline
from dysh.util.files import dysh_data

from dysh.util.selection import Selection
from dysh.spectra.core import mean_tsys

## Helper functions

In [2]:
#from dysh.plot.vegasplot import plot_vegas
def plot_vegas(sdf, scans, title=None, tsys=False, inverse=False, edge=50, ylim=None):
    _fig, ax = plt.subplots(4, 4, sharex="col", sharey="row", gridspec_kw={"hspace": 0, "wspace": 0})

    for r in range(4):
        for c in range(4):
            p = r * 4 + c
            ax[r, c].plot()
            # ax[r,c].set_xlabel(f"{r} {c} {p}")
            for s in scans:
                v1 = sdf.gettp(scan=s, fdnum=p, ifnum=0, plnum=0, calibrate=True, cal=False).timeaverage()
                if tsys:
                    s1 = sdf.gettp(scan=s + 1, fdnum=p, ifnum=0, plnum=0, calibrate=True, cal=False).timeaverage()
                    vs = s1.data / (v1.data - s1.data)
                    if inverse:
                        vs = 1 / vs
                else:
                    vs = v1.data
                ax[r, c].plot(vs[edge:-edge], label=f"{p}")
                if tsys:
                    # note wwe haven't put a proper channel / freq axis
                    nc = len(vs[edge:-edge])
                    fix = np.ones(nc)
                    ax[r, c].plot(fix, color="black")
                # ax[r,c].scatter([0,900],[vs[edge],vs[-edge]], label=f"{p}")

                if ylim is not None:
                    ax[r, c].set_ylim(ylim)
            ax[r, c].legend()
    if title is None:
        plt.suptitle(sdf.filenames()[0])
    else:
        plt.suptitle(title)
    plt.tight_layout()

    if tsys:
        print(f"Showing sky/(vane-sky) for scans={scans},scans+1")
    else:
        print(f"Showing total power for scans={scans}")


In [3]:
def tsys(sdf, scan):
    t=np.zeros(16)
    for f in range(16):
        t[f] = sdf.vanecal(scan=scan,fdnum=f)
    return t

## Data Loading

In [4]:
sdf=GBTOffline('AGBT25B_386_01')
sdf.summary()
pscans = [31,32,35,36,37,38,39,40,41,51,52,53,54,55,56,57,58]


SCAN,OBJECT,VELOCITY,PROC,PROCSEQN,RESTFREQ,# IF,# POL,# INT,# FEED,AZIMUTH,ELEVATION
30,PelicanCenter,0.0,Track,1,88.9095,1,1,60,16,62.0905,45.7890
31,PelicanCenter,0.0,RALongMap,1,88.9095,1,1,277,16,62.3027,46.3188
32,PelicanCenter,0.0,RALongMap,2,88.9095,1,1,277,16,62.5872,47.1576
33,VANE,0.0,Track,1,88.9095,1,1,10,16,63.0282,48.1982
34,SKY,0.0,Track,1,88.9095,1,1,10,16,63.0287,48.1982
35,PelicanCenter,0.0,Track,1,88.9095,1,1,60,16,62.9847,48.4589
36,PelicanCenter,0.0,RALongMap,3,88.9095,1,1,277,16,63.1769,48.9946
37,PelicanCenter,0.0,RALongMap,4,88.9095,1,1,277,16,63.4310,49.8428
38,PelicanCenter,0.0,RALongMap,5,88.9095,1,1,277,16,63.6736,50.6889
39,PelicanCenter,0.0,RALongMap,6,88.9095,1,1,277,16,63.9057,51.5407


In [5]:
pscans = [31,32,36,37,38,39,40,41,51,52,53,54,55,56,57,58]
pscans = [36,37,38,39,40,41,51,52,53,54,55,56,57,58]
tscans = [30,35]

In [6]:
sbp = sdf.gettp(scan=pscans, ifnum=0, plnum=0, fdnum=1)
sbt = sdf.gettp(scan=tscans, ifnum=0, plnum=0, fdnum=1)

In [11]:
sb=sdf.getfs(scan=38, ifnum=0, plnum=0, fdnum=0)
sb.plot()

In [ ]:
sb[0].

In [7]:
sbp.plot().write('junk1.fits', overwrite=True)

In [ ]:
# notice that scan 35 is about 20x weaker
sbt.plot().write('junk2.fits', overwrite=True)

## Plotting the 16 ARGUS beams

Plotting total power and (normalized) Tsys for the 16 beams

In [ ]:
plot_vegas(sdf,[33,34],"TP at 89 GHz")
plt.show()

In [ ]:
# get TSYS values for the VANE/SKY pairs
tsys_33 = tsys(sdf, 33)
tsys_42 = tsys(sdf, 42)
tsys_49 = tsys(sdf, 49)
tsys_59 = tsys(sdf, 59)

In [ ]:
print(tsys_33)
print(tsys_42)
print(tsys_49)
print(tsys_59)


In [ ]:
plot_vegas(sdf,[59,60],"TP at 89 GHz")
plt.show()

Get a spectrum from the Track and OTF. Shouldn't they have diferent freq axes for FS ?

In [ ]:
tp35 = sdf.gettp(scan=35,ifnum=0,fdnum=0,plnum=0).timeaverage()
tp36 = sdf.gettp(scan=36,ifnum=0,fdnum=0,plnum=0).timeaverage()

In [ ]:
tp35.plot()
tp36.plot()

In [ ]:
sdf.getfs

In [ ]:
plot_vegas(sdf,[33],"Normalized Tsys at 89 GHz",tsys=True, ylim=[0.5,2.0])
plt.show()